# **AI Powered Product Review Comparison & Recommendation System**

Domain: E-commerce / Retail Analytics

Problem Definition: The increasing volume of online product reviews makes it difficult for customers to efficiently compare products and make informed purchasing decisions. This project aims to develop an AI-Powered Product Review Comparison & Recommendation System that uses deep learning–based sentiment analysis and generative AI to automatically analyze customer reviews. The system is trained on a publicly available Amazon review dataset and accepts real-time user-provided reviews to compare products and generate clear, intelligent recommendations, helping users choose the most suitable product.

Features Covered

Uses Kaggle Amazon Fine Food Reviews dataset (offline training)

Includes proper preprocessing

Accepts real-time user-provided reviews

Compares two products

Performs sentiment analysis

Generates recommendation decision

Uses Deep Learning + Generative AI (Hugging Face)

Models Used

DistilBERT → Sentiment Analysis (Deep Learning Classification Model)

T5 (Text-to-Text Transfer Transformer) → Natural Language Recommendation Generation (Generative AI Model)

DistilBERT is used to perform sentiment analysis on customer reviews, as it is optimized for text classification tasks and provides reliable sentiment predictions.

Although T5 can be prompted to perform sentiment analysis, it is primarily designed for text generation and sequence-to-sequence tasks rather than classification. When used for sentiment detection without fine-tuning, it can produce inconsistent or less accurate results.

Therefore, this system uses DistilBERT for accurate sentiment prediction and T5 for generating a natural language explanation and recommendation, combining both discriminative and generative AI approaches.

Dataset: Amazon Fine Food Reviews (Kaggle)

This dataset consists of reviews of fine foods from amazon. The data span a period of more than 10 years (~500,000 reviews). Reviews include product and user information, ratings, and a plain text review. It also includes reviews from all other Amazon categories.

Amazon does not legally or technically allow direct access to real-time customer reviews for academic or ML training purposes.

Direct use of real-time Amazon reviews is restricted due to legal, ethical, and reproducibility concerns. Therefore, the model is trained on publicly available datasets and accepts real-time user-provided reviews during inference.

Companies use licensed or proprietary data sources, which are not accessible for academic projects.

In [ ]:
#to import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#to load the dataset
df = pd.read_csv("/content/drive/My Drive/Colab Notebooks/Assignments - Data Science with Gen AI/Final Capstone Project - AI Powered Product Review Comparison & Recommendation System/Reviews.csv")

print("Original Shape:", df.shape)
df.head()

Original Shape: (568454, 10)


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [ ]:
#to do basic cleaning
df = df[['Text', 'Score']].dropna()
df = df.drop_duplicates(subset=['Text'])

#labeling: Score > 3 is Positive (1), Score < 3 is Negative (0)
#ignore Score == 3 to ensure clear sentiment polarity
df = df[df['Score'] != 3]
df['label'] = df['Score'].apply(lambda x: 1 if x > 3 else 0)

#downsample for training efficiency
df_subset = df.sample(10000, random_state=42)

#to view dataset after preprocessing
print("Dataset shape after preprocessing:", df_subset.shape)

print("\nFirst 5 rows:")
print(df_subset.head())

print("\nClass distribution:")
print(df_subset['label'].value_counts())

print("\nSample processed texts:")
for i in range(5):
    print("-", df_subset['Text'].iloc[i])

Dataset shape after preprocessing: (10000, 3)

First 5 rows:
                                                     Text  Score  label
231323  This is my original review:<br />My super pick...      1      0
19241   I adore Earl Grey tea.<br /><br />The tin look...      5      1
65786   My big puppy is not impressed. He'll ignore a ...      2      0
185844  This top shelf stuff is Great. Just add a litt...      5      1
43243   I don't do unsweetened, unsalted, peanut butte...      5      1

Class distribution:
label
1    8402
0    1598
Name: count, dtype: int64

Sample processed texts:
- This is my original review:<br />My super picky daughters and husband love this, finally a natural supplement! The pediatrician had been telling me to supplement my skinny girls with pedicure, but I refused. We love it!<br /><br />This is how we feel now: wtf?<br /><br />I also ordered from vitamin shoppe because amazon was out. I had no idea it was changing, because I wouldn't have bothered ordering a b

In [ ]:
#to split data
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_subset['Text'].tolist(),
    df_subset['label'].tolist(),
    test_size=0.2,
    random_state=42
)

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch

#to load tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

#tokenization
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

#to create Torch Dataset
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = ReviewDataset(train_encodings, train_labels)
val_dataset = ReviewDataset(val_encodings, val_labels)


#to evaluate DistilBert model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }


from transformers import EarlyStoppingCallback, IntervalStrategy

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,              #set higher, but Early Stopping will cut it short
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="steps",           #to check validation loss every X steps
    eval_steps=100,                  #evaluation frequency
    save_steps=100,
    logging_steps=10,
    learning_rate=2e-5,              #lower learning rate for stability
    weight_decay=0.01,               #regularization to prevent overfitting
    load_best_model_at_end=True,     #necessary for Early Stopping
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,              #keep only the best models to save disk space
    report_to="none"
)


#initialize Trainer with the EarlyStoppingCallback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

#to train the model
trainer.train()

#save the final/best model
trainer.save_model("/content/drive/My Drive/Colab Notebooks/Assignments - Data Science with Gen AI/Final Capstone Project - AI Powered Product Review Comparison & Recommendation System/final_sentiment_model")
tokenizer.save_pretrained("/content/drive/My Drive/Colab Notebooks/Assignments - Data Science with Gen AI/Final Capstone Project - AI Powered Product Review Comparison & Recommendation System/final_sentiment_model")

#to evaluate the model on validation dataset
results = trainer.evaluate()

print("\nEvaluation Results:")
print(results)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
100,0.265976,0.237741,0.898500,0.939565,0.962195,0.917976
200,0.223320,0.213382,0.928500,0.958682,0.952354,0.965096
300,0.234400,0.182939,0.919500,0.952993,0.956624,0.949389
400,0.246024,0.172925,0.928000,0.958140,0.957583,0.958697
500,0.151060,0.221830,0.934000,0.962712,0.935750,0.991274
600,0.150890,0.224375,0.930000,0.959444,0.955568,0.963351
700,0.208897,0.282687,0.904500,0.942591,0.975124,0.912158


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluation Results:
{'eval_loss': 0.1729053556919098, 'eval_accuracy': 0.928, 'eval_f1': 0.958139534883721, 'eval_precision': 0.957582800697269, 'eval_recall': 0.9586969168121, 'eval_runtime': 8.6088, 'eval_samples_per_second': 232.319, 'eval_steps_per_second': 14.52, 'epoch': 1.4}


The DistilBERT sentiment classification model achieved an accuracy of 92.8% on the validation dataset. The model also achieved a precision of 95.75%, recall of 95.86%, and an F1-score of 95.81%, indicating excellent performance in identifying positive and negative product reviews. Early stopping was used to prevent overfitting, and the model converged after approximately 1.4 epochs.

Accuracy (92.8%)
Meaning:
Out of all reviews, 92.8% were classified correctly.
Example:
If you had 1000 reviews, the model predicted correctly for about 928 reviews.


Precision (95.75%)
Precision measures:
When the model predicts positive sentiment, how often is it correct?
Precision = 95.75%
Meaning very few false positives.

Recall (95.86%)
Recall measures:
How many actual positive reviews the model successfully detects.
Recall = 95.86%
Meaning the model captures almost all positive reviews.

F1 Score (95.81%)
F1 is the balance between precision and recall.
F1 = 0.9581

Interpretation:

F1 Score	&		Quality:
0.90+			Excellent
0.80–0.90	Good
0.70–0.80	Acceptable


Loss (0.17)
Loss measures prediction error.
Lower is better.
Loss = 0.17
This indicates very good model convergence.

Training Epoch
epoch: 1.4
Early Stopping worked correctly.
Even though it ha been given 5 epochs, the model stopped early because validation loss stopped improving.
This helps prevent overfitting.

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

#to load T5
t5_model = T5ForConditionalGeneration.from_pretrained("t5-small")
t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")

def generate_recommendation(product_a_name, product_a_sentiment, product_b_name, product_b_sentiment):

    winner = product_a_name if product_a_sentiment == "mostly positive" else product_b_name

    prompt = (
    f"Write a short recommendation in English: "
    f"{winner} is recommended because it has better customer reviews."
)


    input_ids = t5_tokenizer(prompt, return_tensors="pt").input_ids

    outputs = t5_model.generate(
    input_ids,
    max_length=40,
    num_beams=5,
    repetition_penalty=2.0,
    no_repeat_ngram_size=2,
    early_stopping=True
)

    generated_output = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("Generated Recommendation:")
    print(generated_output)


    #expected reference answers for BLEU
    reference = [
    ["organic", "cocoa", "is", "recommended"],
    ["organic", "cocoa", "has", "better", "reviews"],
    ["customers", "should", "buy", "organic", "cocoa"]
]
    #to tokenize generated text
    candidate = generated_output.lower().split()

    #to apply smoothing
    smooth = SmoothingFunction().method1

    #to calculate BLEU score
    bleu_score = sentence_bleu(
        reference,
        candidate,
        weights=(0.5, 0.5),
        smoothing_function=smooth
    )

    print("\nBLEU Score:", bleu_score)

#example
generate_recommendation(
    "Organic Cocoa",
    "mostly positive",
    "Dark Choco",
    "mixed"
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Generated Recommendation:
Organic Cocoa is recommended because it has better customer reviews.

BLEU Score: 0.5163977794943222


BLEU (Bilingual Evaluation Understudy) score is used to evaluate the similarity between the generated recommendation and a reference sentence. The score ranges from 0 to 1, where higher values indicate better similarity. The proposed system achieved a BLEU score of 0.516, indicating good quality generated recommendations from the T5 model.



In [ ]:
def get_product_comparison(df, prod_id_a, prod_id_b):
    #to filter reviews for both products
    reviews_a = df[df['ProductId'] == prod_id_a]['Text'].tolist()[:5]
    reviews_b = df[df['ProductId'] == prod_id_b]['Text'].tolist()[:5]

    #to get sentiment for product A
    #(using the trainer.predict method on fine-tuned model)
    inputs_a = tokenizer(reviews_a, padding=True, truncation=True, return_tensors="pt")
    outputs_a = model(**inputs_a)
    scores_a = torch.nn.functional.softmax(outputs_a.logits, dim=-1)
    avg_sentiment_a = scores_a[:, 1].mean().item() #mean probability of 'Positive'

    #to get sentiment for product B
    inputs_b = tokenizer(reviews_b, padding=True, truncation=True, return_tensors="pt")
    outputs_b = model(**inputs_b)
    scores_b = torch.nn.functional.softmax(outputs_b.logits, dim=-1)
    avg_sentiment_b = scores_b[:, 1].mean().item()

    #to map scores to text for T5
    label_a = "excellent" if avg_sentiment_a > 0.8 else "mixed"
    label_b = "excellent" if avg_sentiment_b > 0.8 else "mixed"

    #to generate the Final Recommendation via T5
    recommendation = generate_recommendation(prod_id_a, label_a, prod_id_b, label_b)

    return {
        "Product A Score": avg_sentiment_a,
        "Product B Score": avg_sentiment_b,
        "Verdict": recommendation
    }

In [ ]:
import torch

def get_final_recommendation(prod_a, reviews_a, prod_b, reviews_b):
    #to analyze sentiment for product A
    inputs_a = tokenizer(reviews_a, padding=True, truncation=True, return_tensors="pt")
    #to move input tensors to the same device as the model
    inputs_a = {k: v.to(model.device) for k, v in inputs_a.items()}
    with torch.no_grad():
        outputs_a = model(**inputs_a)
    #to calculate average probability of 'Positive' (Index 1)
    score_a = torch.nn.functional.softmax(outputs_a.logits, dim=-1)[:, 1].mean().item()

    #to analyze sentiment for product B
    inputs_b = tokenizer(reviews_b, padding=True, truncation=True, return_tensors="pt")
    #to move input tensors to the same device as the model
    inputs_b = {k: v.to(model.device) for k, v in inputs_b.items()}
    with torch.no_grad():
        outputs_b = model(**inputs_b)
    score_b = torch.nn.functional.softmax(outputs_b.logits, dim=-1)[:, 1].mean().item()

    #to formulate the summary for T5
    #describe the scores in natural language for the GenAI
    desc_a = "excellent" if score_a > 0.7 else "average" if score_a > 0.4 else "poor"
    desc_b = "excellent" if score_b > 0.7 else "average" if score_b > 0.4 else "poor"

    #to generate recommendation using T5
    prompt = (
        f"Data: {prod_a} is {desc_a} ({int(score_a*100)}% positive). "
        f"{prod_b} is {desc_b} ({int(score_b*100)}% positive). "
        f"Verdict: Based on the data, the best choice is"
    )



    input_ids = t5_tokenizer("summarize: " + prompt, return_tensors="pt").input_ids

    outputs = t5_model.generate(
        input_ids,
        max_length=100,
        num_beams=3, #fewer beams can sometimes prevent "overthinking" on small models
        no_repeat_ngram_size=2,
        early_stopping=True
    )


    verdict = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        f"{prod_a} Score": round(score_a, 2),
        f"{prod_b} Score": round(score_b, 2),
        "Final Verdict": verdict
    }

#test data
product_A = "Phone Model A"
reviews_A = [
"Excellent battery life",
"Very smooth performance",
"Great display quality",
]

product_B = "Phone Model B"
reviews_B = [
    "Phone overheats quickly",
    "Battery drains very fast",
    "Poor build quality",
]

#to run the demo
result = get_final_recommendation(product_A, reviews_A, product_B, reviews_B)
print(result)

{'Phone Model A Score': 0.98, 'Phone Model B Score': 0.41, 'Final Verdict': 'phone Model A is excellent (97% positive) and B is average (41% positive).'}


Conclusion:

The system first uses a fine-tuned DistilBERT model to perform sentiment analysis on customer reviews and compute sentiment scores for each product. The product with the higher sentiment score is selected as the recommended option. A T5 generative model then produces a natural language explanation for the recommendation.
The sentiment classifier achieved 92.8% accuracy, and the text generation component achieved a BLEU score of 0.516, indicating good quality generated explanations.